In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *
from textblob import TextBlob

#pd.set_option("display.max_colwidth', -1)
from time import time
import re
import string
import os

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata

In [ ]:
df_lead_paragraph = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/DateSentiment.pkl", compression=None)

In [ ]:
df_lead_paragraph

In [ ]:
df = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/TimeSentiment.pkl")

In [ ]:
df

In [ ]:
df = pd.to_datetime(df.index)

In [ ]:
df.index = df["Date"]

In [ ]:
df.sort_index(inplace=True)

In [ ]:
df = df["2020-06-01":"2022-06-01"]

In [ ]:
len(df["hours"].unique())

In [ ]:
df['hours']=df['Time'].dt.strftime('%Y-%m-%d %H')

In [ ]:
df['lead_paragraph'].apply(lambda x:len(str(x).split())).sum() #Number Of words in main text

In [ ]:
len(df["minutes"].unique())

In [ ]:
df.Time = pd.to_datetime(df.Time)

In [ ]:
df.index = df.Time.to_period(freq='T').index

In [ ]:
df.Date = df.index

In [ ]:
df['new_date']=df['Time'].dt.strftime('%Y-%m-%d %H')

In [ ]:
df['new_date']=df['new_date'].unique()

In [ ]:
df.index = df.Time #["2020-06-01":]

In [ ]:
df["2020-06-01":].sort_index()

In [ ]:
df_lead_paragraph

In [ ]:
# %run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/Sentiment/Functions.ipynb"

In [ ]:
# New words and values
new_words = {
    'crushes': 100,
    'beats': 5,
    'misses': -5,
    'trouble': -10,
    'falls': -100,
    "dividends":100,
    "dividend":100,
    "earning":100,
    "ipo":100,
    "alliance":100,
    "enhance":100,
    "enhanc":100
}
# Instantiate the sentiment intensity analyzer with the existing lexicon
vader = SentimentIntensityAnalyzer()
# Update the lexicon
vader.lexicon.update(new_words)

In [ ]:
pos_word_list=[]
neu_word_list=[]
neg_word_list=[]
df  = pd.DataFrame()

# Adding new columns to the data frame
df[['compound']] = ''
df[['neg']] = ''
df[['neu']] = ''
df[['pos']] = ''
df[['Sentiment']] = ''
df[["lead_paragraph"]] = ''
df[["polarity"]] = ''
df[["subjectivity"]] = ''
df[["polarityB"]] = ''
df[['GI_Negative']] = ''
df[['GI_Positive']] = ''
df[['GI_Sentiment']] = ''
df[["GI_Polarity"]] = ''
df[["GI_Subjectivity"]] = ''

In [ ]:
import datetime 
import pysentiment2 as ps
hiv4 = ps.HIV4()
for date, row in df_lead_paragraph.T.iteritems():

    sentence = unicodedata.normalize('NFKD', row['lead_paragraph'])#.encode('ascii','ignore').decode('ascii')

    # sentence = str(preprocessing_text(sentence))
    print(date)
    sentence = sentence.lower()
    sentence = sentence.strip("[]")
    sentence = sentence.strip("''")
    df.at[date,"lead_paragraph"] = sentence
    ss = vader.polarity_scores(sentence)

    df.at[date, 'compound'] = ss['compound']
    df.at[date, 'neg'] = ss['neg']
    df.at[date, 'neu'] = ss['neu']
    df.at[date, 'pos'] = ss['pos']
    prc = row['NewsCount']
    d = datetime.timedelta(days = 100)
    formater = "%Y-%m-%d"

    d1 = datetime.datetime.strptime(date,formater)
    date_100 = d1 - d
    date_100=str(date_100)
    mean_prc = df_lead_paragraph.loc[date_100:date]['NewsCount'].mean()
    NewsCount_factor = (prc/mean_prc)
    
    df.at[date, 'NewsCountFactor'] = NewsCount_factor
    if ((ss['pos'] + ss['neg']) != 0):
        number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        if (number != ''):
            df.at[date, 'SentimentOrig'] = (100*(ss['neg'] / (ss['pos'] + ss['neg'])))
            df.at[date, 'Sentiment'] = (100*(ss['neg'] / (ss['pos'] + ss['neg'])))*(NewsCount_factor)

    ############## TEXTBLOB ##############
    blob = TextBlob(sentence)
    df.at[date,"polarity"] = blob.sentiment.polarity
    df.at[date,"subjectivity"] = blob.sentiment.subjectivity
    if (blob.sentiment.polarity > 0 ):
        df.at[date,"polarityB"] = "Positive"
    elif (blob.sentiment.polarity < 0 ):
        df.at[date,"polarityB"] = "Negative"
    elif (blob.sentiment.polarity == 0 ):
        df.at[date,"polarityB"] = "Neutral"


    tokens = hiv4.tokenize(str(sentence))  # text can be tokenized by other ways
                                # however, dict in HIV4 is preprocessed
                                # by the default tokenizer in the library
    score = hiv4.get_score(tokens)

    df.at[date, 'GI_Negative'] = score["Negative"]
    df.at[date, 'GI_Positive'] = score["Positive"]
    df.at[date, 'GI_Polarity'] = score["Polarity"]
    df.at[date, 'GI_Subjectivity'] = score["Subjectivity"]
    if ((score['Positive'] + score['Negative']) != 0):
        number = 100*(score['Positive'] / (score['Positive'] + score['Negative']))
        if (number != ''):
            df.at[date, 'GI_SentimentOrig'] = (100*(score['Negative'] / (score['Positive'] + score['Negative'])))
            df.at[date, 'GI_Sentiment'] = ((100*(score['Negative'] / (score['Positive'] + score['Negative'])))*NewsCount_factor)

df['Sentiment'].apply(np.float)
df['GI_Sentiment'].apply(np.float)
df_lead_paragraph.drop(columns={"lead_paragraph"},inplace=True)
df = df.join(df_lead_paragraph)
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/Sentiment.pkl")

In [ ]:
# del df
# del df_lead_paragraph

In [ ]:
df#["Sentiment"].plot()

In [ ]:
print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))

In [ ]:
df["Negative"].mean()